# Extraction d'informations avec Pandas et Seaborn : les athlètes olympiques

Notebook de l'article [Extraction d'informations avec Pandas et Seaborn](https://moncoachdata.com/blog/extraction-dinformations-avec-pandas-et-seaborn/) (MonCoachData).

Données : *120 ans d'histoire olympique : athlètes et résultats*, fichier `athlete_events.csv` ([page Kaggle](https://www.kaggle.com/heesoo37/120-years-of-olympic-history-athletes-and-results)).

Le code est testé avec Pandas 3 et Seaborn 0.13, et il fonctionne aussi avec Pandas 2 (version de Google Colab).

## 0. Récupérer les données

La cellule suivante télécharge `athlete_events.csv` (271 116 lignes, 15 colonnes) depuis le dépôt GitHub de MonCoachData (copie du fichier compilé par Randi Griffin). Le fichier pèse 41 Mo : il est stocké en deux parties d'environ 20 Mo, que la cellule télécharge puis recolle. Rien à faire : pas de compte ni de jeton.

Si tu préfères la version Kaggle, télécharge `athlete_events.csv` depuis la page du jeu de données et place le fichier à côté du notebook : la cellule ne télécharge rien quand le fichier est déjà là.

In [ ]:
from pathlib import Path
import urllib.request

FICHIER = 'athlete_events.csv'
BASE = 'https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/exploration-de-donnees-statistiques/'
PARTIES = ['athlete_events_partie1.csv', 'athlete_events_partie2.csv']

if not Path(FICHIER).exists():
    with open(FICHIER, 'wb') as sortie:
        for numero, nom in enumerate(PARTIES):
            urllib.request.urlretrieve(BASE + nom, nom)
            lignes = Path(nom).read_bytes().splitlines(keepends=True)
            sortie.writelines(lignes if numero == 0 else lignes[1:])

print(FICHIER, 'prêt :', round(Path(FICHIER).stat().st_size / 1e6, 1), 'Mo')

## 1. Charger les données

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

df = pd.read_csv('athlete_events.csv')
df.shape

## 2. Poids et taille par sport

On travaille sur les hommes (plus de données que sur les femmes). Pour les femmes, change simplement le filtre `df['Sex'] == 'M'` en `df['Sex'] == 'F'`.

In [ ]:
male_df = df[df['Sex'] == 'M']

sport_metrics = male_df.groupby('Sport')[['Weight', 'Height']].agg(['min', 'max', 'mean'])

sport_metrics['Weight'].dropna().sort_values('mean', ascending=False).head(5).round(2)

Sur les 52 tireurs à la corde dont le poids est renseigné, quelle part pèse plus de 80 kg ?

In [ ]:
corde = male_df.loc[male_df['Sport'] == 'Tug-Of-War', 'Weight'].dropna()

# Nombre d'athlètes avec un poids connu, et part de ceux qui pèsent plus de 80 kg
len(corde), float(round((corde > 80).mean(), 2))

### Répartition des poids et des tailles moyens

In [ ]:
mean_weights = sport_metrics['Weight']['mean'].dropna()
mean_heights = sport_metrics['Height']['mean'].dropna()

sns.histplot(mean_weights, kde=True)
plt.xlabel('Poids moyen des hommes par sport (kg)')
plt.ylabel('Nombre de sports')
plt.show()

In [ ]:
sns.histplot(mean_heights, kde=True)
plt.xlabel('Taille moyenne des hommes par sport (cm)')
plt.ylabel('Nombre de sports')
plt.show()

In [ ]:
pd.DataFrame({'poids (kg)': mean_weights.describe(), 'taille (cm)': mean_heights.describe()}).round(1)

### Nuage de points ordonné (avec le nom des deux extrêmes)

In [ ]:
def tracer_classement(valeurs, ylabel):
    """Nuage de points des moyennes triées, avec le nom des deux sports aux extrémités."""
    valeurs = valeurs.sort_values()
    ax = sns.scatterplot(x=range(len(valeurs)), y=valeurs.values)
    ax.set(xlabel='Sports classés du plus petit au plus grand', ylabel=ylabel)
    for rang, cote in ((0, 'left'), (len(valeurs) - 1, 'right')):
        decalage = 10 if cote == 'left' else -10
        ax.annotate(valeurs.index[rang], (rang, valeurs.iloc[rang]),
                    textcoords='offset points', xytext=(decalage, -3), ha=cote)
    plt.show()

tracer_classement(mean_weights, 'Poids moyen (kg)')

In [ ]:
print(mean_weights.sort_values().head(5).round(2).to_string())

In [ ]:
tracer_classement(mean_heights, 'Taille moyenne (cm)')

Pour comparer des kilos et des centimètres, on mesure l'écart à la moyenne en écarts-types :

In [ ]:
# Écart à la moyenne, exprimé en écarts-types (comparable entre kg et cm)
for nom, valeurs in (('poids', mean_weights), ('taille', mean_heights)):
    z = (valeurs - valeurs.mean()) / valeurs.std()
    print(f"{nom} : {z.idxmax()} {z.max():.1f} ; {z.idxmin()} {z.min():.1f}")

In [ ]:
print('Les plus petits :')
print(mean_heights.sort_values().head(5).round(1).to_string())
print('\nLes plus grands :')
print(mean_heights.sort_values(ascending=False).head(5).round(1).to_string())

## 3. Corpulence simplifiée (poids moyen / taille moyenne)

In [ ]:
# Corpulence simplifiée : poids moyen (kg) / taille moyenne (cm)
avg_build = (mean_weights / mean_heights).dropna().sort_values()

sns.lineplot(x=range(len(avg_build)), y=avg_build.values)
plt.xlabel('Sports classés du moins au plus corpulent')
plt.ylabel('Poids / taille (kg par cm)')
plt.show()

In [ ]:
print('Les moins corpulents :')
print(avg_build.head(5).round(3).to_string())
print('\nLes plus corpulents :')
print(avg_build.tail(5)[::-1].round(3).to_string())

## 4. Les sports au cours du temps

In [ ]:
sport_years = male_df.groupby('Sport')['Year'].agg(['min', 'max'])

# Nombre de sports dont la première participation date de chaque année
new_sports = sport_years['min'].value_counts().sort_index()

sns.scatterplot(x=new_sports.index, y=new_sports.values)
plt.xlabel('Année')
plt.ylabel('Nombre de nouveaux sports')
plt.show()

In [ ]:
print(new_sports.to_string())

Sports introduits après 1936, puis sports qui n'ont plus d'athlète masculin depuis avant 2000 :

In [ ]:
sport_years.loc[sport_years['min'] > 1936, 'min'].sort_values(kind='stable')

In [ ]:
sport_years.loc[sport_years['max'] < 2000, 'max'].sort_values(kind='stable')

L'aéronautique : une seule ligne dans le jeu de données.

In [ ]:
df[df['Sport'] == 'Aeronautics'][['Name', 'Team', 'Games', 'Event', 'Medal']]

## 5. À toi de jouer

Y a-t-il des sports où le même pays gagne presque toujours ? Attention : une médaille d'équipe compte une ligne par athlète, donc un sport collectif pèse plus lourd qu'un sport individuel. À toi de voir comment en tenir compte. Refais aussi l'analyse sur les athlètes féminines.

In [ ]:
medaled = df.dropna(subset=['Medal'])
medals = medaled.groupby(['Sport', 'NOC']).size()
total = medals.groupby('Sport').sum()

# Part des médailles du pays le plus titré, pour les sports avec au moins 200 médailles
share_top_country = (medals.groupby('Sport').max() / total)[total >= 200]
share_top_country.sort_values(ascending=False).head(10)